**Deliverable 1**

In [1]:
import requests
import pandas as pd
import os


#Set to True only if I want to rerun the API scraping code.
#It is False by default because the TMDB API key is private.
RUN_SCRAPE = False

if RUN_SCRAPE:

    API_KEY = os.environ.get("TMDB_API_KEY")

    if API_KEY is None:
        raise ValueError("TMDB_API_KEY is not set.")


    url = "https://api.themoviedb.org/3/discover/movie"

    params = {
        "api_key": API_KEY,
        "primary_release_date.gte": "2000-01-01",
        "primary_release_date.lte": "2025-12-31"
    }

    response = requests.get(url, params=params)
    data = response.json()

    print(data.keys()) # movie data is stored in the key results
    print(data["results"][0].keys())


    movie_ids = [movie["id"] for movie in data["results"]]

    raw_movie_data = []

    for movie_id in movie_ids:

        #More specific endpoint so we can get extra data fields
        detail_url = f"https://api.themoviedb.org/3/movie/{movie_id}"

        detail_params = {
            "api_key": API_KEY,
            #Append credits so actor information is included
            "append_to_response": "credits"
        }

        response = requests.get(detail_url, params=detail_params)
        raw_movie_data.append(response.json())


    tmdb_raw = pd.DataFrame(raw_movie_data)

    print(tmdb_raw.columns)
    print(len(tmdb_raw))


#Load the preserved raw dataset used for the actual project
tmdb_raw = pd.read_json("data/raw/tmdb_raw.json")

print("Raw dataset shape:", tmdb_raw.shape)

Raw dataset shape: (26000, 28)


In [2]:
#Get raw data

if RUN_SCRAPE:

    movie_ids = []

    for year in range(2000, 2026):

        year_movie_ids = []

        for page in range(1, 51): #Cycle through 50 pages which is 1000 movies each year

            params = {
                "api_key": API_KEY,
                "primary_release_date.gte": f"{year}-01-01",
                "primary_release_date.lte": f"{year}-12-31",
                "sort_by": "popularity.desc",
                "page": page
            }

            response = requests.get(url, params=params)
            data = response.json()

            year_movie_ids.extend(
                [movie["id"] for movie in data["results"]]
            )

        movie_ids.extend(year_movie_ids)

    print("Total movie IDs:", len(movie_ids)) #There are 26000 movie ids 1000 for each year

In [3]:
#Append credit data from the next endpoint to the raw movie data to get the full raw data of 26,000 movies

if RUN_SCRAPE:

    raw_movie_data = []

    failed_requests = 0

    for i, movie_id in enumerate(movie_ids, start=1): #Go through all the 26000 movie ids

        detail_url = f"https://api.themoviedb.org/3/movie/{movie_id}"

        detail_params = {
            "api_key": API_KEY,
            "append_to_response": "credits"
        }

        try:
            response = requests.get(
                detail_url,
                params=detail_params,
                timeout=30
            )

            if response.status_code == 200:
                raw_movie_data.append(response.json())

            else:
                failed_requests += 1

        except requests.RequestException:
            failed_requests += 1

        if i % 500 == 0:
            print("Processed:", i, "out of", len(movie_ids))


    print("Failed API Requests:", failed_requests)

    tmdb_raw_data = pd.DataFrame(raw_movie_data)

    print("Raw dataset shape:", tmdb_raw_data.shape)

    display(tmdb_raw_data.head())


    #Save the newly scraped data outside data/raw/
    #so the preserved raw file is never overwritten
    tmdb_raw_data.to_json(
        "tmdb_raw.json",
        orient="records"
    )

In [4]:
#Import dataset from saved json file

tmdb_raw = pd.read_json("data/raw/tmdb_raw.json")
tmdb_clean = tmdb_raw.copy()

print(tmdb_raw.shape)
tmdb_raw.head()

(26000, 28)


,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count,credits
0,False,/Ar7QuJ7sJEiC0oP3I8fKBKIQD9u.jpg,"{'id': 1069584, 'name': 'Gladiator Collection'...",103000000,"[{'id': 28, 'name': 'Action'}, {'id': 18, 'nam...",,98,tt0172495,[US],en,...,155,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,What we do in life echoes in eternity.,Gladiator,False,8.231,21486,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
1,False,/m5dA4JZsj78EBTa2D6imWOAebwB.jpg,"{'id': 4246, 'name': 'Scary Movie Collection',...",19000000,"[{'id': 35, 'name': 'Comedy'}]",,4247,tt0175142,[US],en,...,88,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,No mercy. No shame. No sequel.,Scary Movie,False,6.429,8136,"{'cast': [{'adult': False, 'gender': 1, 'id': ..."
2,False,/gTghF8ardwJ48XLHkiapWk3ZmSo.jpg,None,100000000,"[{'id': 80, 'name': 'Crime'}, {'id': 35, 'name...",,107,tt0208092,[GB],en,...,103,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Stealin' stones and breakin' bones.,Snatch,False,7.829,10270,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
3,False,/5oaMV2q0qzxkIW2ukU3lldLu5q2.jpg,"{'id': 86105, 'name': 'American Psycho Collect...",7000000,"[{'id': 53, 'name': 'Thriller'}, {'id': 18, 'n...",,1359,tt0144084,[US],en,...,102,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Killer looks.,American Psycho,False,7.401,12722,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."
4,False,/mZj8EUr6F1x2PWZjKPxaeYd5WRw.jpg,"{'id': 178117, 'name': 'The Emperor's New Groo...",100000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 16, '...",,11688,tt0120917,[US],en,...,78,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,It's all about... ME!,The Emperor's New Groove,False,7.566,7466,"{'cast': [{'adult': False, 'gender': 2, 'id': ..."


In [5]:
#For Deliverable 1, I am going to focus on the columns that will actually be used for my analysis.

analysis_columns = ["id", "title", "release_date", "genres", "budget", "revenue", "popularity", "imdb_id", "credits"]

#In the genres column, which is a list of dictionaries containing genre id and genre name,
#I only plan on using the genre name.

#In the credits column, which is a dictionary containing cast and crew data,
#I only plan on using actor name and actor popularity from the cast data.



#DTYPES
print("Dtypes:\n", tmdb_raw[analysis_columns].dtypes, "\n")

desired_dtypes = {
    "id": "integer",
    "title": "string",
    "release_date": "datetime",
    "genres": "list/object",
    "budget": "numeric",
    "revenue": "numeric",
    "popularity": "numeric",
    "imdb_id": "string",
    "credits": "dictionary/object"
}

print("Desired dtypes: ")

for column, dtype in desired_dtypes.items():
    print(column, ":", dtype)



#MISSING VALUES
print("\n\n")

missing_count = tmdb_raw[analysis_columns].isna().sum()
missing_rate = tmdb_raw[analysis_columns].isna().mean() * 100

print("Missing Count: \n", missing_count)
print("\nMissing Rate (%): \n", missing_rate)
print()


#Check for other common missing-value entries in text columns
text_columns = ["title", "release_date", "imdb_id"]

for column in text_columns:

    print(column)

    print("Blank entries:",
          tmdb_raw[column]
          .astype(str)
          .str.strip()
          .eq("")
          .sum())

    print("Unknown entries:",
          tmdb_raw[column]
          .astype(str)
          .str.strip()
          .str.lower()
          .eq("unknown")
          .sum())

    print()



#Check for missing values inside the genre lists

empty_genres = 0
blank_genres = 0
unknown_genres = 0

for genres in tmdb_raw["genres"]:

    if genres == []:
        empty_genres += 1

    for genre in genres:

        if genre["name"].strip() == "":
            blank_genres += 1

        if genre["name"].strip().lower() == "unknown":
            unknown_genres += 1

print("Empty genre lists:", empty_genres)
print("Blank genre names:", blank_genres)
print("Unknown genre names:", unknown_genres)



#Check for missing values inside the credits dictionary

empty_credits = 0
empty_cast = 0

for credits in tmdb_raw["credits"]:

    if credits == {}:
        empty_credits += 1

    elif credits["cast"] == []:
        empty_cast += 1

print("Empty credit dictionaries:", empty_credits)
print("Empty cast lists:", empty_cast)



#Check actor name and actor popularity values inside the cast data

missing_actor_name = 0
blank_actor_name = 0
unknown_actor_name = 0

missing_actor_popularity = 0
zero_actor_popularity = 0
negative_actor_popularity = 0

for credits in tmdb_raw["credits"]:

    for actor in credits["cast"]:

        #Actor name checks
        if actor["name"] is None:
            missing_actor_name += 1

        else:

            if actor["name"].strip() == "":
                blank_actor_name += 1

            if actor["name"].strip().lower() == "unknown":
                unknown_actor_name += 1


        #Actor popularity checks
        if actor["popularity"] is None:
            missing_actor_popularity += 1

        else:

            if actor["popularity"] == 0:
                zero_actor_popularity += 1

            if actor["popularity"] < 0:
                negative_actor_popularity += 1


print("Missing actor names:", missing_actor_name)
print("Blank actor names:", blank_actor_name)
print("Unknown actor names:", unknown_actor_name)

print()

print("Missing actor popularity:", missing_actor_popularity)
print("Zero actor popularity:", zero_actor_popularity)
print("Negative actor popularity:", negative_actor_popularity)



#Check zero and negative values in the regular numeric columns

numeric_columns = ["id", "budget", "revenue", "popularity"]

print()

for column in numeric_columns:

    print(column)
    print("Zero values:", (tmdb_raw[column] == 0).sum())
    print("Negative values:", (tmdb_raw[column] < 0).sum())
    print()



#DISTINCT VALUES
print("\n\n")

non_nested_columns = ["id", "title", "release_date", "budget", "revenue", "popularity", "imdb_id"]

print("Distinct values:\n",
      tmdb_raw[non_nested_columns].nunique(dropna=False),
      "\n")



#Distinct genre names

genre_names = []

for genres in tmdb_raw["genres"]:

    for genre in genres:

        if genre["name"] not in genre_names:
            genre_names.append(genre["name"])

print("Number of distinct genres:", len(genre_names))
print("Distinct genres:", sorted(genre_names))



#Distinct actor names and actor popularity scores inside credits

actor_names = []
actor_popularities = []

for credits in tmdb_raw["credits"]:

    for actor in credits["cast"]:

        actor_names.append(actor["name"])
        actor_popularities.append(actor["popularity"])

print("Distinct actor names:",
      pd.Series(actor_names).nunique(dropna=False))

print("Distinct actor popularity values:",
      pd.Series(actor_popularities).nunique(dropna=False))



#NUMERIC
print("\n\n")

for column in numeric_columns:

    print(column)

    print("Min:", tmdb_raw[column].min())
    print("Max:", tmdb_raw[column].max())
    print("Range:",
          tmdb_raw[column].max() - tmdb_raw[column].min())

    print()



#Actor popularity is a numerical value inside the credits dictionary

actor_popularity_series = pd.Series(actor_popularities)

print("Actor Popularity")
print("Min:", actor_popularity_series.min())
print("Max:", actor_popularity_series.max())
print("Range:",
      actor_popularity_series.max() - actor_popularity_series.min())



#SPECIFIC DEFECT CHECKS
print("\n\n")



#Sentinel value checks

for column in numeric_columns:

    print(column)

    print("-1 values:",
          (tmdb_raw[column] == -1).sum())

    print("-999 values:",
          (tmdb_raw[column] == -999).sum())

    print()



#Check actor popularity for sentinel values

print("Actor popularity")
print("-1 values:", (actor_popularity_series == -1).sum())
print("-999 values:", (actor_popularity_series == -999).sum())
print()



#Check for total/subtotal rows in title

print("Titles equal to Total:",
      tmdb_raw["title"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("total")
      .sum())

print("Titles equal to Subtotal:",
      tmdb_raw["title"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("subtotal")
      .sum())



#Check for leading zeros in the TMDB movie id

print("TMDB movie IDs with leading zeros:",
      tmdb_raw["id"]
      .astype(str)
      .str.startswith("0")
      .sum())

#IMDb IDs are not checked for leading-zero defects because
#leading zeros are part of the normal IMDb identifier format.



#Check for possible encoding damage in movie titles and actor names

damaged_titles = tmdb_raw[
    tmdb_raw["title"].str.contains("Ã", na=False)
]

print(f"Rows with Ã in title: {len(damaged_titles)} of {len(tmdb_raw)}")

for value in damaged_titles["title"].head(4):
    print(repr(value))

print()


actor_name_series = pd.Series(actor_names)

damaged_actor_names = actor_name_series[
    actor_name_series.str.contains("Ã", na=False)
]

print(f"Actor names with Ã: {len(damaged_actor_names)} of {len(actor_name_series)}")

for value in damaged_actor_names.head(4):
    print(repr(value))



#Check for duplicated TMDB movie ids

print("Duplicate TMDB IDs:",
      tmdb_raw["id"].duplicated().sum())



#Check for duplicated nonblank IMDb ids

valid_imdb_ids = tmdb_raw["imdb_id"].dropna()

valid_imdb_ids = valid_imdb_ids[
    valid_imdb_ids.astype(str).str.strip() != ""
]

print("Duplicate nonblank IMDb IDs:",
      valid_imdb_ids.duplicated().sum())



#Check for release dates stored in more than one format

release_dates = tmdb_raw["release_date"].dropna()

release_dates = release_dates[
    release_dates.astype(str).str.strip() != ""
]

wrong_date_format = release_dates[
    ~release_dates.astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$")
]

print("Release dates with unexpected format:",
      len(wrong_date_format))



#Check for numbers stored as text

for column in numeric_columns:

    print(column)
    print(tmdb_raw[column].map(type).value_counts())
    print()

print("Actor popularity")
print(actor_popularity_series.map(type).value_counts())

Dtypes:
 id                int64
title               str
release_date        str
genres           object
budget            int64
revenue           int64
popularity      float64
imdb_id             str
credits          object
dtype: object 

Desired dtypes: 
id : integer
title : string
release_date : datetime
genres : list/object
budget : numeric
revenue : numeric
popularity : numeric
imdb_id : string
credits : dictionary/object



Missing Count: 
 id                 0
title              0
release_date       0
genres             0
budget             0
revenue            0
popularity         0
imdb_id         2878
credits            0
dtype: int64

Missing Rate (%): 
 id               0.000000
title            0.000000
release_date     0.000000
genres           0.000000
budget           0.000000
revenue          0.000000
popularity       0.000000
imdb_id         11.069231
credits          0.000000
dtype: float64

title
Blank entries: 0
Unknown entries: 4

release_date
Blank entries: 0
Un

The dtypes loaded correctly for the numeric fields and most text fields, although release_date loaded as a string rather than a datetime. The column imdb_id had 2,878 missing values (11.1%) and an additional 375 blank entries. There were 2,026 empty genre lists and 1,760 empty cast lists. Actor names and actor popularity had no standard missing values, although there were 2,018 zero entries for actor popularity. The budget and revenue columns contained 18,205 and 17,862 zero values, respectively. There were also 4 movie titles recorded as "unknown". In terms of specific defects, there were no -1 or -999 sentinel values, total/subtotal rows, leading-zero TMDB IDs, unexpected release-date formats, or numeric values stored as text. There were 424 duplicate TMDB ID occurrences, meaning the 26,000 rows represented 25,576 distinct TMDB movies. Two titles containing the character Ã were inspected for possible encoding damage, but they did not provide clear evidence of mojibake.

In [6]:
#Clean raw data

#Remove duplicate id rows
print("Rows before removing id duplicates:", len(tmdb_clean))
print("Duplicate TMDB IDs:", tmdb_clean["id"].duplicated().sum())

tmdb_clean = tmdb_clean.drop_duplicates(subset="id", keep="first").copy()

print("Rows after removing id duplicates:", len(tmdb_clean))
print("Duplicate TMDB IDs remaining:", tmdb_clean["id"].duplicated().sum())

Rows before removing id duplicates: 26000
Duplicate TMDB IDs: 424
Rows after removing id duplicates: 25576
Duplicate TMDB IDs remaining: 0


In [7]:
#Standardize blank IMDb IDs as missing values

blank_imdb = tmdb_clean["imdb_id"].astype(str).str.strip().eq("")

tmdb_clean.loc[blank_imdb, "imdb_id"] = pd.NA

print("Blank IMDb IDs after:",
      tmdb_clean["imdb_id"].astype(str).str.strip().eq("").sum())

print("Missing IMDb IDs after:",
      tmdb_clean["imdb_id"].isna().sum())

Blank IMDb IDs after: 0
Missing IMDb IDs after: 3195


In [8]:
#Convert release_date from string to datetime

print("Release date dtype before:", tmdb_clean["release_date"].dtype)

tmdb_clean["release_date"] = pd.to_datetime(
    tmdb_clean["release_date"],
    errors="coerce"
)

print("Release date dtype after:", tmdb_clean["release_date"].dtype)
print("Missing release dates after conversion:",
      tmdb_clean["release_date"].isna().sum())

Release date dtype before: str
Release date dtype after: datetime64[us]
Missing release dates after conversion: 0


In [9]:
#Simplify genres from a list of dictionaries to a list of genre names

clean_genres = []

for genres in tmdb_clean["genres"]:

    movie_genres = []

    for genre in genres:
        movie_genres.append(genre["name"])

    clean_genres.append(movie_genres)

tmdb_clean["genres"] = clean_genres

tmdb_clean["genres"].head()

0                         [Action, Drama, Adventure]
1                                           [Comedy]
2                                    [Crime, Comedy]
3                           [Thriller, Drama, Crime]
4    [Adventure, Animation, Comedy, Family, Fantasy]
Name: genres, dtype: object

In [10]:
#Check zero budget and revenue values after duplicate removal

print("Zero budgets:", (tmdb_clean["budget"] == 0).sum())
print("Zero revenues:", (tmdb_clean["revenue"] == 0).sum())

print("Movies with both budget and revenue above 0:",
      ((tmdb_clean["budget"] > 0) & (tmdb_clean["revenue"] > 0)).sum())

#We can only use the movies that have a budget and revenue above zero for this analysis because ROI is what I am planning on examining to interpret results

#Remove movies with unusable budget or revenue values for ROI analysis

print("Rows before financial filtering:", len(tmdb_clean))

tmdb_clean = tmdb_clean[
    (tmdb_clean["budget"] > 0) &
    (tmdb_clean["revenue"] > 0)
].copy()

print("Rows after financial filtering:", len(tmdb_clean))

Zero budgets: 17878
Zero revenues: 17542
Movies with both budget and revenue above 0: 5812
Rows before financial filtering: 25576
Rows after financial filtering: 5812


In [11]:
#Recheck credits after the previous cleaning steps

empty_cast = 0
missing_actor_popularity = 0
zero_actor_popularity = 0

for credits in tmdb_clean["credits"]:

    if credits["cast"] == []:
        empty_cast += 1

    for actor in credits["cast"]:

        if actor["popularity"] is None:
            missing_actor_popularity += 1

        elif actor["popularity"] == 0:
            zero_actor_popularity += 1


print("Movies with empty cast lists:", empty_cast)
print("Missing actor popularity values:", missing_actor_popularity)
print("Zero actor popularity values:", zero_actor_popularity)

#There are still two movies with empty cast lists. They cannot be used for my analysis so I need to remove them
print("Rows before removing empty cast movies:", len(tmdb_clean))

rows_to_keep = []

for credits in tmdb_clean["credits"]:

    if credits["cast"] != []:
        rows_to_keep.append(True)

    else:
        rows_to_keep.append(False)

tmdb_clean = tmdb_clean[rows_to_keep].copy()

print("Rows after removing empty cast movies:", len(tmdb_clean))

Movies with empty cast lists: 2
Missing actor popularity values: 0
Zero actor popularity values: 376
Rows before removing empty cast movies: 5812
Rows after removing empty cast movies: 5810


In [12]:
#Check for and standardize any movies with empty genre lists and titles with "unknown"

empty_genres = 0

for genres in tmdb_clean["genres"]:
    if genres == []:
        empty_genres += 1

print("Empty genre lists remaining:", empty_genres)

clean_genres = []

for genres in tmdb_clean["genres"]:

    if genres == []:
        clean_genres.append(pd.NA)

    else:
        clean_genres.append(genres)

tmdb_clean["genres"] = clean_genres

print("Empty genre lists remaining:",
      sum(genres == [] for genres in tmdb_clean["genres"] if genres is not pd.NA))

print("Missing genre values:",
      tmdb_clean["genres"].isna().sum())


#Unknown titles
print("Unknown titles remaining:",
      tmdb_clean["title"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("unknown")
      .sum())

clean_titles = []

for title in tmdb_clean["title"]:

    if str(title).strip().lower() == "unknown":
        clean_titles.append(pd.NA)

    else:
        clean_titles.append(title)

tmdb_clean["title"] = clean_titles

print("Unknown titles remaining:",
      tmdb_clean["title"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("unknown")
      .sum())

print("Missing titles:",
      tmdb_clean["title"].isna().sum())

Empty genre lists remaining: 7
Empty genre lists remaining: 0
Missing genre values: 7
Unknown titles remaining: 2
Unknown titles remaining: 0
Missing titles: 2


In [13]:
#Verify Clean Data

analysis_columns = ["id", "title", "release_date", "genres", "budget", "revenue", "popularity", "imdb_id", "credits"]

print("Clean dataset shape:", tmdb_clean.shape)
print()


#DTYPES
print("Dtypes:")
print(tmdb_clean[analysis_columns].dtypes)
print()


#MISSING VALUES
print("Missing Count:")
print(tmdb_clean[analysis_columns].isna().sum())

print("\nMissing Rate (%):")
print(tmdb_clean[analysis_columns].isna().mean() * 100)
print()


#CHECK TEXT FIELDS

text_columns = ["title", "imdb_id"]

for column in text_columns:

    print(column)

    print("Blank entries:",
          tmdb_clean[column]
          .astype(str)
          .str.strip()
          .eq("")
          .sum())

    print("Unknown entries:",
          tmdb_clean[column]
          .astype(str)
          .str.strip()
          .str.lower()
          .eq("unknown")
          .sum())

    print()


#CHECK RELEASE DATE

print("Release date dtype:", tmdb_clean["release_date"].dtype)
print("Missing release dates:", tmdb_clean["release_date"].isna().sum())
print()


#CHECK GENRES

empty_genres = 0
genre_names = []

for genres in tmdb_clean["genres"]:

    if isinstance(genres, list):

        if genres == []:
            empty_genres += 1

        for genre in genres:

            if genre not in genre_names:
                genre_names.append(genre)


print("Missing genre values:", tmdb_clean["genres"].isna().sum())
print("Empty genre lists:", empty_genres)
print("Number of distinct genres:", len(genre_names))
print("Distinct genres:", sorted(genre_names))
print()


#CHECK CREDITS / ACTOR DATA

empty_cast = 0

missing_actor_name = 0
blank_actor_name = 0
unknown_actor_name = 0

missing_actor_popularity = 0
zero_actor_popularity = 0
negative_actor_popularity = 0

actor_names = []
actor_popularities = []

for credits in tmdb_clean["credits"]:

    if credits["cast"] == []:
        empty_cast += 1

    for actor in credits["cast"]:

        actor_names.append(actor["name"])
        actor_popularities.append(actor["popularity"])

        #Actor name checks
        if actor["name"] is None:
            missing_actor_name += 1

        else:

            if actor["name"].strip() == "":
                blank_actor_name += 1

            if actor["name"].strip().lower() == "unknown":
                unknown_actor_name += 1


        #Actor popularity checks
        if actor["popularity"] is None:
            missing_actor_popularity += 1

        else:

            if actor["popularity"] == 0:
                zero_actor_popularity += 1

            if actor["popularity"] < 0:
                negative_actor_popularity += 1


print("Empty cast lists:", empty_cast)

print("Missing actor names:", missing_actor_name)
print("Blank actor names:", blank_actor_name)
print("Unknown actor names:", unknown_actor_name)

print("Missing actor popularity:", missing_actor_popularity)
print("Zero actor popularity:", zero_actor_popularity)
print("Negative actor popularity:", negative_actor_popularity)

print()


#CHECK NUMERIC VALUES

numeric_columns = ["id", "budget", "revenue", "popularity"]

for column in numeric_columns:

    print(column)

    print("Zero values:", (tmdb_clean[column] == 0).sum())
    print("Negative values:", (tmdb_clean[column] < 0).sum())

    print("Min:", tmdb_clean[column].min())
    print("Max:", tmdb_clean[column].max())
    print("Range:",
          tmdb_clean[column].max() - tmdb_clean[column].min())

    print()


#ACTOR POPULARITY NUMERIC CHECK

actor_popularity_series = pd.Series(actor_popularities)

print("Actor Popularity")
print("Min:", actor_popularity_series.min())
print("Max:", actor_popularity_series.max())
print("Range:",
      actor_popularity_series.max() - actor_popularity_series.min())
print()


#DISTINCT VALUES

non_nested_columns = [
    "id",
    "title",
    "release_date",
    "budget",
    "revenue",
    "popularity",
    "imdb_id"
]

print("Distinct values:")
print(tmdb_clean[non_nested_columns].nunique(dropna=False))
print()

print("Distinct actor names:",
      pd.Series(actor_names).nunique(dropna=False))

print("Distinct actor popularity values:",
      actor_popularity_series.nunique(dropna=False))

print()


#SPECIFIC DEFECT CHECKS

#Sentinel values
for column in numeric_columns:

    print(column)
    print("-1 values:", (tmdb_clean[column] == -1).sum())
    print("-999 values:", (tmdb_clean[column] == -999).sum())
    print()


print("Actor popularity")
print("-1 values:", (actor_popularity_series == -1).sum())
print("-999 values:", (actor_popularity_series == -999).sum())
print()


#Total/subtotal rows

print("Titles equal to Total:",
      tmdb_clean["title"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("total")
      .sum())

print("Titles equal to Subtotal:",
      tmdb_clean["title"]
      .astype(str)
      .str.strip()
      .str.lower()
      .eq("subtotal")
      .sum())


#Leading-zero TMDB IDs

print("TMDB movie IDs with leading zeros:",
      tmdb_clean["id"]
      .astype(str)
      .str.startswith("0")
      .sum())


#Possible encoding damage

damaged_titles = tmdb_clean[
    tmdb_clean["title"].astype(str).str.contains("Ã", na=False)
]

print("Titles containing Ã:", len(damaged_titles))

for value in damaged_titles["title"].head(4):
    print(repr(value))

print()


#Duplicate TMDB IDs

print("Duplicate TMDB IDs:",
      tmdb_clean["id"].duplicated().sum())


#Duplicate usable IMDb IDs

valid_imdb_ids = tmdb_clean["imdb_id"].dropna()

valid_imdb_ids = valid_imdb_ids[
    valid_imdb_ids.astype(str).str.strip() != ""
]

print("Duplicate nonblank IMDb IDs:",
      valid_imdb_ids.duplicated().sum())


#NUMBERS STORED AS TEXT

for column in numeric_columns:

    print(column)
    print(tmdb_clean[column].map(type).value_counts())
    print()

print("Actor popularity")
print(actor_popularity_series.map(type).value_counts())

Clean dataset shape: (5810, 28)

Dtypes:
id                       int64
title                      str
release_date    datetime64[us]
genres                  object
budget                   int64
revenue                  int64
popularity             float64
imdb_id                    str
credits                 object
dtype: object

Missing Count:
id               0
title            2
release_date     0
genres           7
budget           0
revenue          0
popularity       0
imdb_id         26
credits          0
dtype: int64

Missing Rate (%):
id              0.000000
title           0.034423
release_date    0.000000
genres          0.120482
budget          0.000000
revenue         0.000000
popularity      0.000000
imdb_id         0.447504
credits         0.000000
dtype: float64

title
Blank entries: 0
Unknown entries: 0

imdb_id
Blank entries: 0
Unknown entries: 0

Release date dtype: datetime64[us]
Missing release dates: 0

Missing genre values: 7
Empty genre lists: 0
Number of di

The missing title, genres, and imdb_id values dont matter as much because for the roi and actor popularity analysis I am doing, they are not as important. Additionally, when I merge with the imdb dataset, I hope to obtain the missing imdb_id values.

In [14]:
#Simplify credits to only the actor information needed for analysis

clean_actors = []

for credits in tmdb_clean["credits"]:

    movie_actors = []

    for actor in credits["cast"]:

        actor_info = {
            "name": actor["name"],
            "popularity": actor["popularity"]
        }

        movie_actors.append(actor_info)

    clean_actors.append(movie_actors)


#Add the simplified actors column
tmdb_clean["actors"] = clean_actors


#Remove the original credits column
tmdb_clean = tmdb_clean.drop(columns=["credits"])


#Check the result
print(tmdb_clean["actors"].head())
print("\nColumns:")
print(tmdb_clean.columns)

0    [{'name': 'Russell Crowe', 'popularity': 4.959...
1    [{'name': 'Anna Faris', 'popularity': 3.636900...
2    [{'name': 'Jason Statham', 'popularity': 16.71...
3    [{'name': 'Christian Bale', 'popularity': 5.02...
4    [{'name': 'David Spade', 'popularity': 2.1212}...
Name: actors, dtype: object

Columns:
Index(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres',
       'homepage', 'id', 'imdb_id', 'origin_country', 'original_language',
       'original_title', 'overview', 'popularity', 'poster_path',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'softcore', 'spoken_languages', 'status',
       'tagline', 'title', 'video', 'vote_average', 'vote_count', 'actors'],
      dtype='str')


In [15]:
#Create movie-level actor popularity variables

top_actor_names = []
max_actor_popularities = []
top_3_actor_popularities = []

for actors in tmdb_clean["actors"]:

    #Sort the actors from highest to lowest popularity
    sorted_actors = sorted(
        actors,
        key=lambda actor: actor["popularity"],
        reverse=True
    )

    #Most popular actor
    top_actor_names.append(sorted_actors[0]["name"])
    max_actor_popularities.append(sorted_actors[0]["popularity"])

    #Average popularity of the top 3 actors
    top_3 = sorted_actors[:3]

    total_popularity = 0

    for actor in top_3:
        total_popularity += actor["popularity"]

    average_top_3 = total_popularity / len(top_3)

    top_3_actor_popularities.append(average_top_3)


#Add the new columns to the cleaned dataset
tmdb_clean["top_actor_name"] = top_actor_names
tmdb_clean["max_actor_popularity"] = max_actor_popularities
tmdb_clean["top_3_actor_popularity"] = top_3_actor_popularities


#Check the new columns
tmdb_clean[
    ["title", "top_actor_name", "max_actor_popularity", "top_3_actor_popularity"]
].head()

,title,top_actor_name,max_actor_popularity,top_3_actor_popularity
0,Gladiator,Russell Crowe,4.9597,4.402733
1,Scary Movie,Lochlyn Munro,4.4736,3.770633
2,Snatch,Jason Statham,16.7122,10.520733
3,American Psycho,Willem Dafoe,5.8196,4.797167
4,The Emperor's New Groove,John Goodman,3.6030,3.038533


In [16]:
#Create ROI column

tmdb_clean["roi"] = (
    (tmdb_clean["revenue"] - tmdb_clean["budget"])
    / tmdb_clean["budget"]
)

tmdb_clean[["title", "budget", "revenue", "roi"]].head()

,title,budget,revenue,roi
0,Gladiator,103000000,465516248,3.519575
1,Scary Movie,19000000,278023280,13.632804
2,Snatch,100000000,83557872,-0.164421
3,American Psycho,7000000,34269748,3.895678
4,The Emperor's New Groove,100000000,169327687,0.693277


In [17]:
#Trim down tmdb_clean so that I only have the columns I am going to use for the analysis

columns_to_keep = [
    "id",
    "imdb_id",
    "title",
    "release_date",
    "genres",
    "budget",
    "revenue",
    "roi",
    "top_actor_name",
    "max_actor_popularity",
    "top_3_actor_popularity"
]

tmdb_clean = tmdb_clean[columns_to_keep].copy()

print(tmdb_clean.columns)
print(tmdb_clean.shape)

#Save cleaned TMDB data as JSON

tmdb_clean.to_json(
    "tmdb_clean.json",
    orient="records",
    date_format="iso"
)

Index(['id', 'imdb_id', 'title', 'release_date', 'genres', 'budget', 'revenue',
       'roi', 'top_actor_name', 'max_actor_popularity',
       'top_3_actor_popularity'],
      dtype='str')
(5810, 11)


**Deliverable 2**

| Variable name | Type | Units | Factor levels | Valid range | Missingness | Notes |
|---|---|---|---|---|---|---|
| `id` | int | N/A | N/A | Positive integers (`> 0`) | 0 missing | Unique TMDB movie identifier. Stored as an integer but used only as an identifier. Duplicate IDs were removed during cleaning. |
| `imdb_id` | string | N/A | N/A | Valid IMDb title identifier format | 26 missing | IMDb identifier used to connect TMDB movies to the IMDb ratings dataset. Missing means no usable IMDb ID was available after cleaning. |
| `title` | string | N/A | N/A | Any nonblank movie title | 2 missing | Movie title from TMDB. Values recorded as `"unknown"` were standardized to missing during cleaning. Different movies may share the same title, so `title` is not a unique identifier. |
| `release_date` | datetime | N/A | N/A | Valid calendar dates from 2000-01-01 through 2025-12-31 | 0 missing | Movie release date from TMDB. Converted from string to datetime during cleaning. |
| `genres` | list | N/A | Action, Adventure, Animation, Comedy, Crime, Documentary, Drama, Family, Fantasy, History, Horror, Music, Mystery, Romance, Science Fiction, TV Movie, Thriller, War, Western | One or more valid TMDB genre categories | 7 missing | Each movie may belong to multiple genres, so the cleaned value is stored as a list of genre names. Empty genre lists were standardized to missing during cleaning. |
| `budget` | int | dollars | N/A | Greater than 0 | 0 missing | Production budget reported by TMDB. Movies with a budget of 0 were removed during cleaning because ROI cannot be calculated with a zero budget. |
| `revenue` | int | dollars | N/A | Greater than 0 | 0 missing | Box-office revenue reported by TMDB. Movies with a revenue of 0 were removed during cleaning because they could not support the ROI analysis. |
| `roi` | float | unitless ratio | N/A | Greater than -1 | 0 missing | Calculated as `(revenue - budget) / budget`. A value of 1 represents a 100% return, 0 represents break-even, and negative values represent a loss. |
| `top_actor_name` | string | N/A | N/A | Any nonblank actor name | 0 missing | Name of the actor with the highest TMDB popularity score in the movie's cast. |
| `max_actor_popularity` | float | unitless score | N/A | Greater than or equal to 0 | 0 missing | Highest TMDB popularity score among all actors in the movie's cast. Used as the primary measure of whether a movie includes a high-profile actor. |
| `top_3_actor_popularity` | float | unitless score | N/A | Greater than or equal to 0 | 0 missing | Average TMDB popularity score of the three most popular actors in the movie's cast. If a movie has fewer than three actors, the average uses all available actors. |


Documentation check: No cleaned variables were found to contain values that clearly contradicted their documented formats or valid ranges. Data-quality issues identified in the raw data, including duplicate movie IDs, zero budget/revenue values, blank IMDb IDs, and empty genre/cast data, were addressed during cleaning.


**Deliverable 3**

| # | Column | Change made | Rows/cells affected | Why | What is lost |
|---|---|---|---|---|---|
| 1 | `id` | Removed duplicate movie rows based on TMDB `id`, keeping the first occurrence | 424 rows | The cleaned dataset should have one row per movie, and duplicate IDs violated that intended grain | Duplicate copies of 424 movie records were removed |
| 2 | `imdb_id` | Standardized blank or whitespace-only IMDb IDs to missing values (`pd.NA`) | 369 cells | Blank strings did not represent valid IMDb identifiers and needed to be treated consistently as missing data | The distinction between an empty string and a missing value is lost |
| 3 | `release_date` | Converted release dates from strings to datetime values | 25,576 cells | Datetime values are easier to validate, filter, and use in time-based analysis than raw text strings | The original text formatting of the dates is lost |
| 4 | `genres` | Simplified each genre from a dictionary containing an ID and name to a list of genre names only | 25,576 cells | The analysis only needs the genre labels, and the nested dictionary structure was unnecessarily complex | TMDB genre ID values and the original dictionary structure are lost |
| 5 | `budget`, `revenue` | Removed movies with a budget or revenue less than or equal to 0 | 19,764 rows | The ROI analysis was restricted to movies with positive reported budget and revenue values so zero financial values would not be treated as usable financial information | 19,764 movie records are excluded, so the analysis only represents movies with positive reported budget and revenue |
| 6 | `credits` | Removed movies with empty cast lists | 2 rows | Movies without any cast information cannot be used to measure actor popularity for the analysis | 2 movie records are excluded from the cleaned dataset |
| 7 | `genres` | Standardized empty genre lists (`[]`) to missing values (`pd.NA`) | 7 cells | An empty list indicates that no usable genre information was available and should be represented consistently as missing data | The distinction between an empty list and a missing value is lost |
| 8 | `title` | Standardized titles recorded as `"unknown"` to missing values (`pd.NA`) | 2 cells | `"unknown"` is not a meaningful movie title and should be treated as missing information | The literal `"unknown"` placeholder values are lost |
| 9 | `credits` → `actors` | Replaced the full `credits` structure with a simplified list containing only each actor's name and TMDB popularity score | 5,810 rows | The analysis only needs actor names and popularity scores, so the additional cast and crew information was unnecessary | Crew information and other cast metadata from `credits` are removed from the cleaned dataset |
| 10 | `top_actor_name`, `max_actor_popularity` | Identified the actor with the highest TMDB popularity score in each movie and recorded the actor's name and popularity score | 5,810 rows | A movie-level measure was needed to represent the presence of a high-profile actor in each cast | Nothing |
| 11 | `top_3_actor_popularity` | Calculated the mean TMDB popularity of the three most popular actors in each movie's cast | 5,810 rows | Using the top three actors provides a broader measure of cast star power than considering only one actor | Nothing |
| 12 | `roi` | Calculated ROI as `(revenue - budget) / budget` for each movie | 5,810 rows | ROI is the main outcome variable for the research question because it measures financial return relative to the movie's production budget | Nothing |
| 13 | Dataset-wide | Removed columns that were not needed for the planned analysis, keeping only the 11 analysis and identification variables | 122,010 cells (21 columns across 5,810 rows) | The cleaned dataset was reduced to the variables needed for the research question and later IMDb join | Information contained in the 21 removed columns is not present in the cleaned dataset but remains available in the raw data |

**Deliverable 4**

In [18]:
#Count exact duplicate rows by converting each row to a JSON string (I have to do this because some collumns in the raw file are lists)
raw_rows_as_json = tmdb_raw.to_json(
    orient="records",
    lines=True
).splitlines()

exact_duplicates = pd.Series(raw_rows_as_json).duplicated().sum()


#Count rows that repeat an existing TMDB ID
duplicate_id_rows = tmdb_raw.duplicated(
    subset="id",
    keep="first"
).sum()


#Near-duplicates are duplicate IDs that are not exact duplicate rows
near_duplicates = duplicate_id_rows - exact_duplicates

#rows removed for other reasons (I removed the rows where budget or revenue <= 0, and the two rows that had an empty cast list)
rows_removed_other = 19764 + 2
print("Rows removed for other reasons:", rows_removed_other)

#Find which columns were dropped
columns_dropped = []

for column in tmdb_raw.columns:
    if column not in tmdb_clean.columns:
        columns_dropped.append(column)

#Find which columns were added
columns_added = []

for column in tmdb_clean.columns:
    if column not in tmdb_raw.columns:
        columns_added.append(column)


print("Rows in raw file: ", len(tmdb_raw))
print("Exact duplicate rows:", exact_duplicates)
print("Duplicate TMDB ID rows:", duplicate_id_rows)
print("Near-duplicate rows:", near_duplicates)
print("Rows removed for other reasons:", rows_removed_other)
print("Rows in clean file: ", len(tmdb_clean))
print("Columns in raw file: ", len(tmdb_raw.columns))
print("Columns dropped: ", len(columns_dropped))
print("Columns added: ", len(columns_added))
print("Columns in clean file: ", len(tmdb_clean.columns))

Rows removed for other reasons: 19766
Rows in raw file:  26000
Exact duplicate rows: 424
Duplicate TMDB ID rows: 424
Near-duplicate rows: 0
Rows removed for other reasons: 19766
Rows in clean file:  5810
Columns in raw file:  28
Columns dropped:  21
Columns added:  4
Columns in clean file:  11


Rows in raw file:                      26000

Rows removed as exact duplicates:      424

Rows removed as near-duplicates:       0

Rows removed for other reasons:        19766   

Rows in cleaned file:                  5810


Columns in raw file:                   28

Columns dropped:                       21  Why: These columns were not needed for my planned ROI and actor-based analysis. 

Columns added:                         4

Columns in cleaned file:               11

**Deliverable 5**

This dataset is movie information for movies released between 2000 - 2025. The data was retrieved through The Movie Database (TMDB), which is a community-built movie database containing movie, tv, and actor information. TMDB maintains this data so users can have a free, detailed information about movies and etnertainment. I collected 1,000 of the most pouplar movies from each year using TMDB's API. My cleaned dataset contains movie ids, titles, release dates, genres, production budgets, box-office revenue, ROI, and measures of actor popularity. However, movies that did not have any reported budget or financial data were excluded from the cleaned dataset, leaving data for 5,810 movies. The dataset can be used to examine the relationship between actor popularity and movie ROI. But it can't be used to definititvely prove that popularity actors cause a higher or lower return on investment for all movies.